# Day 3 — Contract Clause Search (REFERENCE / worked solution)

**Pattern:** Retrieval-augmented generation (RAG) with FAISS  
**Domain:** Legal  
**Course reference:** Lab 3 (part 1)

---

> ⚠️ **This is the filled-in reference notebook.** Every TODO from `Day_03_contract_clause_search.ipynb` is implemented here so you can check your own work or unstick yourself. Try the scaffold first — the learning is in *writing* it. Use this only when you're stuck or want to compare approaches.

## Setup

Run once. If you only want the core lab (Steps 1–4), the first three packages are enough; the rest power the LangChain / LlamaIndex stretch goals at the bottom.

In [ ]:
# --- Setup: install the core packages for the from-scratch embedding lab (Steps 1-4) ---
# Install CURRENT versions (no old pins) so this works on modern Python (3.12+).
# IMPORTANT: do NOT pin "numpy<2.0" on Python 3.13/3.14 -- NumPy 1.x has no wheel there,
# and forcing it breaks faiss, which shows up as an error in Step 3.
# `sentence-transformers` pulls in PyTorch automatically.
%pip install -q -U sentence-transformers faiss-cpu "numpy>=2"

# After this finishes, RESTART THE KERNEL once (Kernel -> Restart), then Run All from the top.
# That guarantees the kernel loads the freshly installed numpy/faiss instead of stale ones.
#
# The LangChain / LlamaIndex "stretch" cells at the bottom are optional and target an older
# framework API -- skip them for now; this track has dedicated LangChain projects later.
print("Setup complete. Now restart the kernel, then run Steps 1-4 top to bottom.")

## Step 1 — Load and chunk the contracts

Each `.txt` in `contracts/` starts with a 6-line metadata header (SOURCE / FILER / EXHIBIT TYPE / URL / RETRIEVED) followed by a `====` separator and then the contract body. We split that header off and keep only the body, then chunk it into ~500-word windows with 50-word overlap, preferring paragraph and sentence boundaries so we never cut mid-word.

In [ ]:
import os
import re
import glob

CONTRACTS_DIR = "contracts"  # relative to Day03/


def _strip_header(raw: str) -> str:
    """Drop the SEC EDGAR metadata header (SOURCE/FILER/... + '====' separator); return the body."""
    if "=" * 10 in raw:
        return raw.split("=" * 10, 1)[1].lstrip("=").lstrip()
    return raw


def load_documents(folder_path: str) -> list:
    """Read every .txt file and return a list of dicts: {'text', 'source', 'page'}.

    These are plain-text exports, so there are no real page numbers; we set page=1.
    (If you fed in PDFs via pypdf you'd emit one dict per page with the real page index.)
    """
    documents = []
    for path in sorted(glob.glob(os.path.join(folder_path, "*.txt"))):
        with open(path, "r", encoding="utf-8") as f:
            raw = f.read()
        documents.append({
            "text": _strip_header(raw),
            "source": os.path.basename(path),
            "page": 1,
        })
    return documents


def chunk_text(text: str, chunk_size: int = 500, overlap: int = 50) -> list:
    """Split text into ~chunk_size-word windows with `overlap` words of carry-over.

    Word-based (not true token-based) for simplicity — close enough for a starter lab and
    it never cuts mid-word. We pack whole paragraphs until we'd exceed chunk_size, which
    keeps related sentences together.
    """
    paragraphs = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    chunks, current, current_len = [], [], 0

    for para in paragraphs:
        words = para.split()
        if current_len + len(words) > chunk_size and current:
            chunks.append(" ".join(current))
            # carry the last `overlap` words into the next chunk for context continuity
            carry = current[-overlap:] if overlap else []
            current, current_len = list(carry), len(carry)
        current.extend(words)
        current_len += len(words)
        # a single paragraph bigger than chunk_size: flush it on its own
        while current_len > chunk_size:
            chunks.append(" ".join(current[:chunk_size]))
            current = current[chunk_size - overlap:]
            current_len = len(current)

    if current:
        chunks.append(" ".join(current))
    return chunks


# --- build the flat list of chunk dicts with metadata preserved ---
documents = load_documents(CONTRACTS_DIR)
all_chunks = []
for doc in documents:
    for i, chunk in enumerate(chunk_text(doc["text"])):
        all_chunks.append({
            "text": chunk,
            "source": doc["source"],
            "page": doc["page"],
            "chunk_id": i,
        })

avg_len = sum(len(c["text"].split()) for c in all_chunks) / max(len(all_chunks), 1)
print(f"Documents : {len(documents)}")
print(f"Chunks    : {len(all_chunks)}")
print(f"Avg words : {avg_len:.0f} per chunk")
print("\nExample chunk:")
print(" source:", all_chunks[0]["source"])
print(" text  :", all_chunks[0]["text"][:200], "...")

## Step 2 — Embed the chunks

Each chunk becomes a vector. We use the local `all-MiniLM-L6-v2` sentence-transformer — 384 dimensions, free, fast, no API key. The same model **must** embed both the corpus and the queries, or the scores are meaningless.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

chunk_texts = [c["text"] for c in all_chunks]
embeddings = embedder.encode(chunk_texts, show_progress_bar=True, convert_to_numpy=True)
embeddings = embeddings.astype("float32")  # FAISS wants float32

print("Embeddings shape:", embeddings.shape)  # (n_chunks, 384)


def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


# Sanity check: a confidentiality-ish query should sit closer to an NDA chunk than to a random one.
probe = embedder.encode("obligation to keep information confidential", convert_to_numpy=True)
sims = [(cosine(probe, emb), all_chunks[i]["source"]) for i, emb in enumerate(embeddings)]
sims.sort(reverse=True)
print("\nClosest chunk to the confidentiality probe:", sims[0])
print("Farthest chunk:", sims[-1])

## Step 3 — Build a FAISS index

`IndexFlatIP` does exact inner-product search. After L2-normalizing the vectors, inner product **equals cosine similarity** — which is what we want for semantic closeness.

In [ ]:
import faiss

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)

# Normalize a COPY so our stored `embeddings` stays intact if we want raw vectors later.
index_vectors = embeddings.copy()
faiss.normalize_L2(index_vectors)  # in-place
index.add(index_vectors)

print("index.ntotal:", index.ntotal, "(should match chunk count:", len(all_chunks), ")")

## Step 4 — Search

Embed the query with the *same* model, normalize it the *same* way, ask FAISS for the top-k nearest chunks, then look up the metadata so every hit carries its source filename and score.

In [ ]:
def search(query: str, k: int = 5) -> list:
    """Return the top-k chunks for a natural-language query as (chunk_dict, score) tuples."""
    q = embedder.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(q)
    scores, idxs = index.search(q, k)
    results = []
    for score, idx in zip(scores[0], idxs[0]):
        if idx == -1:
            continue
        results.append((all_chunks[idx], float(score)))
    return results


queries = [
    "non-compete period longer than one year",
    "restrictions on hiring our employees after the contract ends",
    "who owns the IP if we co-develop something",
    "limitation of liability and indemnification",
    "obligation to keep information confidential",
]

for q in queries:
    print(f"\n=== {q} ===")
    for chunk, score in search(q, k=3):
        snippet = " ".join(chunk["text"].split())[:180]
        print(f"[{chunk['source']}] (cos={score:.3f}) {snippet}...")

## Step 5 — Where retrieval-only breaks

Notice the synonym win: *"restrictions on hiring our employees after the contract ends"* surfaces non-solicitation language ("shall not solicit, induce, or attempt to engage any employee") with **zero keyword overlap**. That's the magic of embeddings.

But retrieval is necessary, not sufficient — the cell below shows you still have to *read* 200+ words to extract the actual answer. Tomorrow (Day 4) we feed these chunks to an LLM and ask it to synthesize a cited answer. That is full RAG.

In [ ]:
q = "who owns the IP if we co-develop something"
top = search(q, k=1)[0]
chunk, score = top
print(f"Query: {q}")
print(f"Top chunk from {chunk['source']} (cos={score:.3f}), {len(chunk['text'].split())} words:\n")
print(chunk["text"])
print("\n--- The clause is in there, but the lawyer still has to read the whole thing.")
print("--- That extraction gap is the motivation for Day 4 (generation on top of retrieval).")

## Self-check — answers

- **Chunk too small (50 tokens)?** Each chunk loses surrounding context, so a clause that spans several sentences gets split and no single chunk carries the full meaning — retrieval returns fragments. **Too large (2000)?** Each vector averages over many unrelated topics, blurring the embedding so similarity scores flatten and you retrieve loosely-relevant walls of text.
- **Why normalize before inner-product search?** Inner product on raw vectors rewards long vectors (large magnitude) regardless of direction. After L2 normalization every vector has length 1, so inner product = cosine of the angle = pure semantic similarity, independent of magnitude.
- **Can one retrieve-top-k call combine three clauses across two contracts?** Not reliably. Top-k returns the k chunks closest to the *query* vector; if the answer requires synthesizing scattered clauses, some needed chunks may rank below k or look dissimilar to the query phrasing. You need multi-query retrieval, re-ranking, or an LLM that reasons over a larger retrieved set — again, Day 4 territory.

---
## Stretch A — LangChain

The same Steps 1–4 pipeline collapsed to ~10 lines. Watch the flagged default: `RecursiveCharacterTextSplitter`'s `chunk_size` is in **characters**, not words/tokens — so `500` here is a *much* smaller window than your from-scratch chunker.

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

docs = DirectoryLoader(
    "contracts", glob="*.txt",
    loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"},
).load()

# WATCH OUT: chunk_size is in CHARACTERS here, not tokens.
chunks = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50).split_documents(docs)

lc_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
db = FAISS.from_documents(chunks, lc_embeddings)  # normalization + index handled internally

queries = [
    "non-compete period longer than one year",
    "restrictions on hiring our employees after the contract ends",
    "who owns the IP if we co-develop something",
]
for q in queries:
    print(f"\n=== {q} ===")
    for doc, score in db.similarity_search_with_score(q, k=3):
        src = doc.metadata.get("source", "?")
        print(f"[{src}] (dist={score:.3f}) {doc.page_content[:160].strip()}...")

## Stretch B — LlamaIndex

Same pipeline again. `SentenceSplitter`'s `chunk_size` is in **tokens** and it splits on sentence boundaries — closest of the three to what Step 1 asked you to hand-roll. We set `Settings.llm = None` so it does pure retrieval (no accidental OpenAI calls).

In [ ]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

Settings.embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
Settings.llm = None  # retrieval only today
Settings.node_parser = SentenceSplitter(chunk_size=500, chunk_overlap=50)  # TOKENS, sentence-aware

documents_li = SimpleDirectoryReader("contracts", required_exts=[".txt"]).load_data()
li_index = VectorStoreIndex.from_documents(documents_li)
retriever = li_index.as_retriever(similarity_top_k=3)

queries = [
    "non-compete period longer than one year",
    "restrictions on hiring our employees after the contract ends",
    "who owns the IP if we co-develop something",
]
for q in queries:
    print(f"\n=== {q} ===")
    for node in retriever.retrieve(q):
        src = node.metadata.get("file_name", "?")
        print(f"[{src}] (score={node.score:.3f}) {node.text[:160].strip()}...")

## What the framework comparison teaches you

- **From scratch (Steps 1–4):** you control chunking, normalization, index type, and distance metric — more lines, but every layer is debuggable.
- **LangChain:** ~10 lines; `chunk_size` is in **characters** — a hidden default that silently changes retrieval quality.
- **LlamaIndex:** ~10 lines; `SentenceSplitter` measures **tokens** and respects sentence boundaries — closest to the hand-rolled intent.

**Interview takeaway:** lead with the from-scratch mechanics ("I chunk with overlap, normalize, use cosine via inner product..."), *then* "in production I'd reach for LlamaIndex/LangChain to avoid reinventing it." Knowing the primitives is what lets you explain *why* the three approaches return different chunks for the same query.